# Stage 1 — PDF Ingestion Pipeline
### Space Flight AI — Dataset Builder

This notebook:
1. Parses all PDFs in a folder using Docling
2. Detects scanned vs digital PDFs automatically
3. Exports clean markdown to `parsed_docs.json`
4. Feeds parsed text into an LLM to generate structured decision+reasoning training pairs
5. Saves final dataset as `training_pairs.jsonl` ready for fine-tuning

---
**Folder structure expected:**
```
project/
├── pdfs/                  ← drop all your downloaded PDFs here
│   ├── nasa_apollo11.pdf
│   ├── orbital_mechanics.pdf
│   └── ...
├── parsed_docs.json       ← auto generated after Step 2
└── training_pairs.jsonl   ← auto generated after Step 4
```

## Step 0 — Install Dependencies

In [1]:
# Run once — installs all required packages
!pip install docling pymupdf openai tqdm --quiet

# If running on university H100 cluster, you may need:
# !pip install docling pymupdf openai tqdm --quiet --break-system-packages

## Step 1 — Imports and Config

In [2]:
import os
import json
import pymupdf as fitz
from pathlib import Path
from tqdm import tqdm
from docling.document_converter import DocumentConverter
from docling.datamodel.pipeline_options import PdfPipelineOptions
from openai import OpenAI            # using OpenAI-compatible API

# ─────────────────────────────────────────────
# CONFIG — edit these paths before running
# ─────────────────────────────────────────────
PDF_FOLDER = "./pdfs"                 # folder where you drop your PDFs
PARSED_OUTPUT    = "./parsed_docs.json"     # intermediate parsed markdown
TRAINING_OUTPUT  = "./training_pairs.jsonl" # final fine-tuning dataset

# LLM API config — used to generate training pairs from parsed text
# You can use any OpenAI-compatible endpoint
# Options: OpenAI GPT-4, local Ollama, Together AI, etc.
LLM_API_KEY      = "your-gpt-api-key"
LLM_BASE_URL = "https://api.openai.com/v1"
LLM_MODEL    = "gpt-4o-mini"

# How many training pairs to generate per document chunk
PAIRS_PER_CHUNK  = 5

# Chunk size in characters — how much text fed to LLM at once
# 3000 chars ≈ ~750 tokens — safe for most models
CHUNK_SIZE       = 3000

print("✓ Config loaded")
print(f"  PDF folder     : {PDF_FOLDER}")
print(f"  Parsed output  : {PARSED_OUTPUT}")
print(f"  Training output: {TRAINING_OUTPUT}")

/home/jovyan/.local/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✓ Config loaded
  PDF folder     : ./pdfs
  Parsed output  : ./parsed_docs.json
  Training output: ./training_pairs.jsonl


## Step 2 — Parse All PDFs with Docling

Automatically detects whether each PDF is:
- **Digital** (modern NASA/ESA docs) → parsed directly
- **Scanned** (Apollo-era docs) → OCR enabled automatically

In [3]:
# ADD THESE at the top of the cell with other imports
from docling.datamodel.base_models import InputFormat
from docling.document_converter import PdfFormatOption
from docling.datamodel.pipeline_options import PdfPipelineOptions

def needs_ocr(pdf_path: str) -> bool:
    """
    Detect if a PDF is scanned (image-based) vs digital.
    If the first page has almost no extractable text → it's scanned.
    """
    try:
        doc = fitz.open(pdf_path)
        first_page_text = doc[0].get_text().strip()
        doc.close()
        return len(first_page_text) < 50
    except Exception:
        return True  # if in doubt, use OCR


def get_converter(ocr: bool) -> DocumentConverter:
    if ocr:
        options = PdfPipelineOptions(do_ocr=True)
        return DocumentConverter(
            format_options={InputFormat.PDF: PdfFormatOption(pipeline_options=options)}
        )
    return DocumentConverter()


def parse_pdf(pdf_path: str) -> dict:
    """
    Parse a single PDF.
    Returns dict with filename, ocr_used, and markdown text.
    """
    ocr = needs_ocr(pdf_path)
    converter = get_converter(ocr)
    result = converter.convert(pdf_path)
    markdown = result.document.export_to_markdown()
    return {
        "filename": Path(pdf_path).name,
        "ocr_used": ocr,
        "char_count": len(markdown),
        "text": markdown
    }


def parse_all_pdfs(folder: str) -> dict:
    """
    Parse every PDF in the folder.
    Returns dict keyed by filename stem.
    Skips files that fail gracefully.
    """
    pdf_files = list(Path(folder).glob("*.pdf"))

    if not pdf_files:
        print(f"No PDFs found in {folder}")
        print("    Drop your PDF files there and re-run this cell.")
        return {}

    print(f"Found {len(pdf_files)} PDF(s) — starting parse...\n")
    parsed = {}

    for pdf in tqdm(pdf_files, desc="Parsing PDFs"):
        try:
            result = parse_pdf(str(pdf))
            parsed[pdf.stem] = result
            ocr_tag = "[OCR]" if result["ocr_used"] else "[digital]"
            print(f"  ✓ {pdf.name} {ocr_tag} — {result['char_count']:,} chars")
        except Exception as e:
            print(f"  ✗ {pdf.name} — FAILED: {e}")

    return parsed


# ── RUN ──
os.makedirs(PDF_FOLDER, exist_ok=True)
parsed_docs = parse_all_pdfs(PDF_FOLDER)

if parsed_docs:
    with open(PARSED_OUTPUT, "w") as f:
        json.dump(parsed_docs, f, indent=2)
    print(f"\n✓ Saved parsed docs → {PARSED_OUTPUT}")
    print(f"  Total documents parsed: {len(parsed_docs)}")
    total_chars = sum(d['char_count'] for d in parsed_docs.values())
    print(f"  Total characters     : {total_chars:,}")

Found 23 PDF(s) — starting parse...



Parsing PDFs:   0%|          | 0/23 [00:00<?, ?it/s][INFO] 2026-08-26 08:31:24,818 [RapidOCR] base.py:23: Using engine_name: torch
[INFO] 2026-08-26 08:31:24,820 [RapidOCR] device_config.py:64: Using GPU device with ID: 0
[INFO] 2026-08-26 08:31:24,821 [RapidOCR] download_file.py:68: Initiating download: https://www.modelscope.cn/models/RapidAI/RapidOCR/resolve/v3.9.2/torch/PP-OCRv6/det/PP-OCRv6_det_small.pth
[INFO] 2026-08-26 08:31:26,829 [RapidOCR] download_file.py:82: Download size: 9.77MB
[INFO] 2026-08-26 08:31:29,463 [RapidOCR] download_file.py:95: Successfully saved to: /home/jovyan/.local/lib/python3.10/site-packages/rapidocr/models/PP-OCRv6_det_small.pth
[INFO] 2026-08-26 08:31:29,467 [RapidOCR] main.py:50: Using /home/jovyan/.local/lib/python3.10/site-packages/rapidocr/models/PP-OCRv6_det_small.pth
[INFO] 2026-08-26 08:31:30,920 [RapidOCR] base.py:23: Using engine_name: torch
[INFO] 2026-08-26 08:31:30,921 [RapidOCR] device_config.py:64: Using GPU device with ID: 0
[INFO] 202

  ✓ 19630002820.pdf [digital] — 69,433 chars


[INFO] 2026-08-26 08:33:30,519 [RapidOCR] base.py:23: Using engine_name: torch
[INFO] 2026-08-26 08:33:30,520 [RapidOCR] device_config.py:64: Using GPU device with ID: 0
[INFO] 2026-08-26 08:33:30,521 [RapidOCR] download_file.py:60: File exists and is valid: /home/jovyan/.local/lib/python3.10/site-packages/rapidocr/models/ch_ptocr_mobile_v2.0_cls_mobile.pth
[INFO] 2026-08-26 08:33:30,522 [RapidOCR] main.py:50: Using /home/jovyan/.local/lib/python3.10/site-packages/rapidocr/models/ch_ptocr_mobile_v2.0_cls_mobile.pth
[INFO] 2026-08-26 08:33:30,579 [RapidOCR] base.py:23: Using engine_name: torch
[INFO] 2026-08-26 08:33:30,580 [RapidOCR] device_config.py:64: Using GPU device with ID: 0
[INFO] 2026-08-26 08:33:30,592 [RapidOCR] download_file.py:60: File exists and is valid: /home/jovyan/.local/lib/python3.10/site-packages/rapidocr/models/PP-OCRv6_rec_small.pth
[INFO] 2026-08-26 08:33:30,593 [RapidOCR] main.py:50: Using /home/jovyan/.local/lib/python3.10/site-packages/rapidocr/models/PP-OCRv

  ✓ 19630011222.pdf [digital] — 710,524 chars


...

  ✓ Tutorial_Advanced Rocket Design - Kerbal Space Program Wiki.pdf [digital] — 13,616 chars


[INFO] 2026-08-26 09:13:32,746 [RapidOCR] device_config.py:64: Using GPU device with ID: 0
[INFO] 2026-08-26 09:13:32,748 [RapidOCR] download_file.py:60: File exists and is valid: /home/jovyan/.local/lib/python3.10/site-packages/rapidocr/models/ch_ptocr_mobile_v2.0_cls_mobile.pth
[INFO] 2026-08-26 09:13:32,748 [RapidOCR] main.py:50: Using /home/jovyan/.local/lib/python3.10/site-packages/rapidocr/models/ch_ptocr_mobile_v2.0_cls_mobile.pth
[INFO] 2026-08-26 09:13:32,800 [RapidOCR] base.py:23: Using engine_name: torch
[INFO] 2026-08-26 09:13:32,801 [RapidOCR] device_config.py:64: Using GPU device with ID: 0
[INFO] 2026-08-26 09:13:32,814 [RapidOCR] download_file.py:60: File exists and is valid: /home/jovyan/.local/lib/python3.10/site-packages/rapidocr/models/PP-OCRv6_rec_small.pth
[INFO] 2026-08-26 09:13:32,814 [RapidOCR] main.py:50: Using /home/jovyan/.local/lib/python3.10/site-packages/rapidocr/models/PP-OCRv6_rec_small.pth

Parsing PDFs:  96%|█████████▌| 22/23 [42:11<00:14, 14.70s/it]

  ✓ Tutorial_Basic SSTO Design - Kerbal Space Program Wiki.pdf [digital] — 13,814 chars


[INFO] 2026-08-26 09:13:36,450 [RapidOCR] base.py:23: Using engine_name: torch
[INFO] 2026-08-26 09:13:36,451 [RapidOCR] device_config.py:64: Using GPU device with ID: 0
[INFO] 2026-08-26 09:13:36,452 [RapidOCR] download_file.py:60: File exists and is valid: /home/jovyan/.local/lib/python3.10/site-packages/rapidocr/models/ch_ptocr_mobile_v2.0_cls_mobile.pth
[INFO] 2026-08-26 09:13:36,452 [RapidOCR] main.py:50: Using /home/jovyan/.local/lib/python3.10/site-packages/rapidocr/models/ch_ptocr_mobile_v2.0_cls_mobile.pth
[INFO] 2026-08-26 09:13:36,505 [RapidOCR] base.py:23: Using engine_name: torch
[INFO] 2026-08-26 09:13:36,505 [RapidOCR] device_config.py:64: Using GPU device with ID: 0
[INFO] 2026-08-26 09:13:36,517 [RapidOCR] download_file.py:60: File exists and is valid: /home/jovyan/.local/lib/python3.10/site-packages/rapidocr/models/PP-OCRv6_rec_small.pth
[INFO] 2026-08-26 09:13:36,518 [RapidOCR] main.py:50: Using /home/jovyan/.local/lib/python3.10/site-packages/rapidocr/models/PP-OCRv

  ✓ preview-9780080470542_A25023383.pdf [OCR] — 57,705 chars

✓ Saved parsed docs → ./parsed_docs.json
  Total documents parsed: 23
  Total characters     : 3,178,960


## Step 3 — Inspect Parsed Output

Quick sanity check before feeding to LLM — make sure the text looks clean.

In [4]:
# Load parsed docs if not already in memory
with open(PARSED_OUTPUT) as f:
    parsed_docs = json.load(f)

# Show summary table
print(f"{'Document':<40} {'OCR':<8} {'Chars':>10}")
print("-" * 62)
for name, doc in parsed_docs.items():
    ocr_tag = "yes" if doc["ocr_used"] else "no"
    print(f"{name[:40]:<40} {ocr_tag:<8} {doc['char_count']:>10,}")

# Preview first 1000 chars of the first document
print("\n" + "=" * 62)
print("PREVIEW — first document, first 1000 characters:")
print("=" * 62)
first_doc = list(parsed_docs.values())[0]
print(first_doc["text"][:1000])

Document                                 OCR           Chars
--------------------------------------------------------------
19630002820                              no           69,433
19630011222                              no          710,524
19650019871                              yes          47,129
19660016018                              no          122,277
19670022649                              no           26,370
19670026467                              no          120,368
19680007746                              no          187,830
19680010999                              no          172,541
19740004369                              yes         238,434
19830016258                              yes         448,755
20030000844                              no           36,281
20080009584                              no           59,029
20160012009                              no           42,645
Bate, Mueller, and White - Fundamentals  yes         672,918
Introduction to Orbita

## Step 4 — Generate Training Pairs with LLM

Each parsed document is split into chunks.
Each chunk is sent to the LLM with a structured prompt.
The LLM returns decision+reasoning pairs in JSON format.

**Output format per pair:**
```json
{
  "mission_phase": "Gravity Turn Ascent",
  "situation": "Vehicle at 15km altitude, velocity 350 m/s, dynamic pressure 28 kPa",
  "decision": "Reduce throttle to 70%, maintain current pitch angle",
  "chain_of_thought": "Dynamic pressure approaching max-Q...",
  "theory_reference": "NASA ascent profile guidelines — Max-Q management",
  "source_document": "nasa_apollo11.pdf"
}
```

In [5]:
# ── LLM CLIENT SETUP ──
client = OpenAI(
    api_key=LLM_API_KEY,
    base_url=LLM_BASE_URL
)


SYSTEM_PROMPT = """
You are a space flight expert building a training dataset for an AI rocket pilot.

You will receive a chunk of text from a space document — this could be theory, 
equations, procedures, mission reports, design notes, or operational guidelines.

Your job is to extract as many USEFUL training pairs as possible from whatever 
is in the text. Be creative and flexible — not every pair needs to be a direct 
flight decision. The AI pilot needs to understand space deeply.

Generate pairs in ANY of these formats depending on what the text contains:

FORMAT A — Flight Decision (when text has operational content)
  situation     : telemetry or scenario requiring a decision
  decision      : what the pilot should do
  chain_of_thought : physics reasoning behind the decision
  theory_reference : principle involved

FORMAT B — Concept Understanding (when text has theory/principles)
  situation     : "Explain [concept] and when it applies during flight"
  decision      : clear explanation of the concept
  chain_of_thought : deeper physics — why it works this way
  theory_reference : source principle or equation

FORMAT C — What-If Reasoning (when text describes constraints or limits)
  situation     : "What happens if [limit is exceeded or condition changes]?"
  decision      : consequence and correct response
  chain_of_thought : physical reasoning for the consequence
  theory_reference : the governing principle

FORMAT D — Procedure Knowledge (when text has step-by-step content)
  situation     : "What is the correct sequence for [procedure]?"
  decision      : the steps in correct order with rationale
  chain_of_thought : why each step must happen in this order
  theory_reference : the operational principle

Always set mission_phase to the most relevant flight phase this knowledge applies to.
If knowledge applies to multiple phases, pick the most specific one.

Mission phases to choose from:
Pre-Launch, Ignition, Liftoff, Gravity Turn, Max-Q, Stage Separation,
Orbital Insertion, Circularization, Hohmann Transfer, Plane Change,
Rendezvous, Docking, Station Keeping, Deorbit Burn, Re-entry,
Landing Burn, Touchdown, Mission Planning, Abort, General Orbital Mechanics

Output ONLY a valid JSON array. No preamble, no explanation, no markdown fences.
Each element must have exactly these keys:
  mission_phase, situation, decision, chain_of_thought, theory_reference

If the chunk is completely irrelevant to spaceflight return []
"""


def chunk_text(text: str, chunk_size: int = CHUNK_SIZE) -> list:
    """Split text into overlapping chunks to avoid cutting mid-concept."""
    chunks = []
    step = int(chunk_size * 0.85)  # 15% overlap between chunks
    for i in range(0, len(text), step):
        chunk = text[i:i + chunk_size]
        if len(chunk) > 200:  # skip tiny trailing chunks
            chunks.append(chunk)
    return chunks


def generate_pairs_from_chunk(chunk: str, source_name: str) -> list:
    """
    Send one text chunk to the LLM.
    Returns a list of structured training pair dicts.
    """
    user_prompt = f"""
Source document: {source_name}

Text chunk:
---
{chunk}
---

Generate up to {PAIRS_PER_CHUNK} decision+reasoning training pairs from this text.
Return only a JSON array.
"""
    try:
        response = client.chat.completions.create(
            model=LLM_MODEL,
            messages=[
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user",   "content": user_prompt}
            ],
            temperature=0.3,   # low temp = more factual, less creative
            max_tokens=2000
        )

        raw = response.choices[0].message.content.strip()

        # Strip markdown fences if LLM adds them despite instructions
        raw = raw.replace("```json", "").replace("```", "").strip()

        pairs = json.loads(raw)

        # Inject source document name into each pair
        for p in pairs:
            p["source_document"] = source_name

        return pairs

    except json.JSONDecodeError as e:
        print(f"JSON parse error: {e}")
        return []
    except Exception as e:
        print(f"LLM error: {e}")
        return []


print("✓ LLM pipeline functions defined — ready to run Step 5")

✓ LLM pipeline functions defined — ready to run Step 5


## Step 5 — Run the Full Pipeline

This loops through every parsed document, chunks it, calls the LLM, and saves pairs.

**Cost estimate:** ~$0.002 per chunk with GPT-4o. A 100-page PDF ≈ 50 chunks ≈ $0.10 per document.

In [6]:
all_pairs = []
failed_chunks = 0

for doc_name, doc_data in tqdm(parsed_docs.items(), desc="Documents"):
    text   = doc_data["text"]
    chunks = chunk_text(text)

    print(f"\n📄 {doc_name} — {len(chunks)} chunks")

    doc_pairs = []
    for i, chunk in enumerate(tqdm(chunks, desc=f"  Chunks", leave=False)):
        pairs = generate_pairs_from_chunk(chunk, doc_name)

        if pairs:
            doc_pairs.extend(pairs)
        else:
            failed_chunks += 1

    all_pairs.extend(doc_pairs)
    print(f"  ✓ Generated {len(doc_pairs)} pairs from this document")

# ── SAVE AS JSONL ──
# JSONL = one JSON object per line — standard format for LLM fine-tuning
with open(TRAINING_OUTPUT, "w") as f:
    for pair in all_pairs:
        f.write(json.dumps(pair) + "\n")

print("\n" + "=" * 50)
print(f"✓ Pipeline complete")
print(f"  Total pairs generated : {len(all_pairs)}")
print(f"  Failed chunks skipped : {failed_chunks}")
print(f"  Saved to              : {TRAINING_OUTPUT}")

Documents:   0%|          | 0/23 [00:00<?, ?it/s]


📄 19630002820 — 28 chunks



  Chunks:  82%|████████▏ | 23/28 [01:50<00:25,  5.08s/it]

Documents:   4%|▍         | 1/23 [02:17<50:29, 137.72s/it][A

  ✓ Generated 105 pairs from this document

📄 19630011222 — 279 chunks



  Chunks:  84%|████████▎ | 233/279 [14:18<01:19,  1.72s/it]

Documents:   9%|▊         | 2/23 [18:36<3:41:18, 632.30s/it][A

...

  ✓ Generated 25 pairs from this document

📄 Tutorial_Basic SSTO Design - Kerbal Space Program Wiki — 6 chunks



Documents:  96%|█████████▌| 22/23 [1:09:24<00:57, 57.80s/it]

  ✓ Generated 25 pairs from this document

📄 preview-9780080470542_A25023383 — 23 chunks



Documents: 100%|██████████| 23/23 [1:10:25<00:00, 183.72s/it]

  ✓ Generated 35 pairs from this document

✓ Pipeline complete
  Total pairs generated : 3020
  Failed chunks skipped : 654
  Saved to              : ./training_pairs.jsonl


## Step 6 — Inspect and Validate Training Pairs

Before fine-tuning, always check what was generated.
Look for: blank fields, hallucinated physics, wrong phase names.

In [7]:
# Load and display sample pairs
with open(TRAINING_OUTPUT) as f:
    loaded_pairs = [json.loads(line) for line in f]

print(f"Total training pairs loaded: {len(loaded_pairs)}\n")

# Distribution of mission phases
from collections import Counter
phase_counts = Counter(p.get("mission_phase", "unknown") for p in loaded_pairs)
print("Mission phase distribution:")
for phase, count in phase_counts.most_common():
    bar = "█" * (count // max(1, max(phase_counts.values()) // 20))
    print(f"  {phase:<35} {count:>4}  {bar}")

# Show 3 random pairs in full
import random
print("\n" + "=" * 60)
print("SAMPLE PAIRS (3 random):")
for pair in random.sample(loaded_pairs, min(3, len(loaded_pairs))):
    print("\n" + "-" * 60)
    print(f"Phase      : {pair.get('mission_phase')}")
    print(f"Situation  : {pair.get('situation')}")
    print(f"Decision   : {pair.get('decision')}")
    print(f"Reasoning  : {pair.get('chain_of_thought')}")
    print(f"Theory ref : {pair.get('theory_reference')}")
    print(f"Source     : {pair.get('source_document')}")

Total training pairs loaded: 3020

Mission phase distribution:
  General Orbital Mechanics            900  ████████████████████
  Orbital Insertion                    574  ████████████
  Mission Planning                     359  ███████
  Rendezvous                           285  ██████
  Re-entry                             238  █████
  Liftoff                              204  ████
  Hohmann Transfer                     160  ███
  Pre-Launch                            44  
  Ignition                              32  
  Deorbit Burn                          32  
  Ascent                                27  
  Landing                               24  
  Launch                                21  
  Landing Burn                          19  
  Stage Separation                      11  
  Max-Q                                 10  
  Transfer                              10  
  Station Keeping                       10  
  Plane Change                           9  
  Abort                  

## Step 7 — Convert to Fine-Tuning Format

Convert raw pairs into the prompt/response format that QLoRA fine-tuning expects.
Uses the special tokens defined for this project.

**Output format:**
```
<|mission_phase|>Gravity Turn<|telemetry|>altitude: 15000m...<|reasoning|>...<|decision|>Reduce throttle to 70%
```

In [8]:
FINETUNE_OUTPUT = "./finetune_ready.jsonl"

def format_for_finetuning(pair: dict) -> dict:
    """
    Convert a raw training pair into prompt/response format.
    Handles all 4 pair formats — decision, concept, what-if, procedure.
    """
    mission_phase = pair.get("mission_phase", "General Orbital Mechanics")
    situation     = pair.get("situation", "")
    reasoning     = pair.get("chain_of_thought", "")
    theory        = pair.get("theory_reference", "")
    decision      = pair.get("decision", "")

    # Skip malformed pairs
    if not situation.strip() or not decision.strip():
        return None

    prompt = (
        f"<|mission_phase|>{mission_phase}\n"
        f"<|situation|>{situation}\n"
        f"Provide your reasoning and response."
    )

    response = (
        f"<|reasoning|>{reasoning}\n"
        f"<|theory_ref|>{theory}\n"
        f"<|decision|>{decision}"
    )

    return {
        "messages": [
            {"role": "system",    "content": "You are an AI rocket pilot with deep knowledge of orbital mechanics and space flight operations. Analyse the situation and provide detailed reasoning for your response."},
            {"role": "user",      "content": prompt},
            {"role": "assistant", "content": response}
        ],
        "source": pair.get("source_document", "")
    }

# Load pairs
with open(TRAINING_OUTPUT) as f:
    loaded_pairs = [json.loads(line) for line in f if line.strip()]

# Convert — filter out None (malformed pairs)
finetune_data = [format_for_finetuning(p) for p in loaded_pairs]
finetune_data = [d for d in finetune_data if d is not None]

# Save
with open(FINETUNE_OUTPUT, "w") as f:
    for item in finetune_data:
        f.write(json.dumps(item) + "\n")

print(f"✓ Fine-tune ready dataset saved → {FINETUNE_OUTPUT}")
print(f"  Raw pairs loaded  : {len(loaded_pairs)}")
print(f"  Valid pairs saved : {len(finetune_data)}")
print(f"  Filtered out      : {len(loaded_pairs) - len(finetune_data)} malformed")

print("\nSAMPLE FORMATTED ENTRY:")
sample = finetune_data[0]
for msg in sample["messages"]:
    print(f"\n[{msg['role'].upper()}]")
    print(msg["content"])

✓ Fine-tune ready dataset saved → ./finetune_ready.jsonl
  Raw pairs loaded  : 3020
  Valid pairs saved : 3020
  Filtered out      : 0 malformed

SAMPLE FORMATTED ENTRY:

[SYSTEM]
You are an AI rocket pilot with deep knowledge of orbital mechanics and space flight operations. Analyse the situation and provide detailed reasoning for your response.

[USER]
<|mission_phase|>Mission Planning
<|situation|>What are the earth orbit launch criteria for lunar trajectories?
Provide your reasoning and response.

[ASSISTANT]
<|reasoning|>The launch criteria must take into account the vehicle's performance capabilities and the mission's trajectory needs to ensure successful lunar insertion.
<|theory_ref|>Orbital Mechanics
<|decision|>Define the launch criteria based on the physical conditions of the launch vehicle and specific mission requirements.


## Step 8 — Dataset Health Check

Final checks before handing off to the fine-tuning notebook.
Flags any pairs with empty fields, too-short reasoning, or missing theory references.

In [9]:
issues = []
REQUIRED_FIELDS = ["mission_phase", "situation", "decision", "chain_of_thought", "theory_reference"]
MIN_REASONING_CHARS = 50   # reduced from 100 — concept pairs can be shorter

for i, pair in enumerate(loaded_pairs):
    for field in REQUIRED_FIELDS:
        val = pair.get(field, "")
        if not str(val).strip():
            issues.append(f"Pair {i}: missing '{field}'")

    if len(pair.get("chain_of_thought", "")) < MIN_REASONING_CHARS:
        issues.append(f"Pair {i}: reasoning too short — {len(pair.get('chain_of_thought',''))} chars")

if issues:
    print(f"Found {len(issues)} issues:\n")
    for issue in issues[:20]:
        print(f"  - {issue}")
    if len(issues) > 20:
        print(f"  ... and {len(issues) - 20} more")
else:
    print(f"✓ All {len(loaded_pairs)} pairs passed health check")
    print(f"  Dataset is ready for 02_finetune_qwen.ipynb")

from collections import Counter
phase_counts = Counter(p.get("mission_phase", "unknown") for p in loaded_pairs)
avg_reasoning_len = sum(len(p.get("chain_of_thought", "")) for p in loaded_pairs) / max(1, len(loaded_pairs))

print(f"\nDataset stats:")
print(f"  Total pairs          : {len(loaded_pairs)}")
print(f"  Unique sources       : {len(set(p.get('source_document') for p in loaded_pairs))}")
print(f"  Unique phases        : {len(phase_counts)}")
print(f"  Avg reasoning length : {avg_reasoning_len:.0f} chars")
print(f"\nTop phases:")
for phase, count in phase_counts.most_common(10):
    bar = "█" * (count // max(1, max(phase_counts.values()) // 20))
    print(f"  {phase:<35} {count:>4}  {bar}")

✓ All 3020 pairs passed health check
  Dataset is ready for 02_finetune_qwen.ipynb

Dataset stats:
  Total pairs          : 3020
  Unique sources       : 23
  Unique phases        : 42
  Avg reasoning length : 155 chars

Top phases:
  General Orbital Mechanics            900  ████████████████████
  Orbital Insertion                    574  ████████████
  Mission Planning                     359  ███████
  Rendezvous                           285  ██████
  Re-entry                             238  █████
  Liftoff                              204  ████
  Hohmann Transfer                     160  ███
  Pre-Launch                            44  
  Ignition                              32  
  Deorbit Burn                          32  


---
## ✓ Pipeline Complete

**Output files:**
| File | Description |
|---|---|
| `parsed_docs.json` | Raw markdown from all PDFs |
| `training_pairs.jsonl` | Structured decision+reasoning pairs |
| `finetune_ready.jsonl` | Formatted for QLoRA fine-tuning |

**Next step:** Open `02_finetune_qwen.ipynb` and load `finetune_ready.jsonl`

**Recommended minimum dataset size before fine-tuning:**
- 500 pairs — basic results
- 2,000 pairs — good generalization  
- 5,000+ pairs — strong domain knowledge